In [1]:
# Get raw data
with open('input/18.txt', 'r') as f:
    rawinput = f.read().strip()

In [2]:
import numpy as np

In [3]:
# Part 1
plan = [[j[0],int(j[1]),j[2]] 
        for i in rawinput.split('\n') 
        for j in [i.split()]]
directions = {'U':(-1,0),'R':(0,1),'D':(1,0),'L':(0,-1)}

lagoon = [[0,0]]
for d,s,_ in plan:
    for i in range(s):
        lagoon += [[sum(j) for j in zip(lagoon[-1], directions[d])]]
limits = [[min(i)-1, max(i)+2] for i in zip(*lagoon)]

layout = np.ones([j-i for i,j in limits], dtype=int)
layout[*[[j-i[-1][0] for j in i[:-1]] for i in zip(*lagoon,limits)]] = 2
layout = np.pad(layout[1:-1,1:-1],1)

evolve = lambda x: np.where((x == 1) 
                            & np.any(np.stack(((z:=np.pad(x,1))[:-2,1:-1], 
                                               z[2:,1:-1], 
                                               z[1:-1,:-2], 
                                               z[1:-1,2:]))==0, 
                                     axis=0), 0, x)
while np.any(layout != (layout:=evolve(layout))):  pass

np.sum(layout > 0).item()

42317

In [4]:
# Part 2
lagoon = [[0,0]]
for _,_,h in plan:
    d,s = 'RDLU'[int(h[7])], int(h[2:7],16)
    lagoon += [[i+j*s for i,j in zip(lagoon[-1], directions[d])]]
    
posconv, gaps = [],[]
for i in zip(*lagoon):
    st = sorted({*sum([[j,j+1] for j in {*i}],[])})
    posconv += [{l:k for k,l in enumerate(st)}]
    gaps += [[k-l for k,l in zip(st[1:],st)]]
lagoon = [[posconv[j][k] for j,k in enumerate(i)] 
          for i in lagoon]

limits = [[min(i), max(i)+1] for i in zip(*lagoon)]
layout = np.ones([*zip(*limits)][1], dtype=int)
for (yf,xf),(yt,xt) in zip(lagoon[:-1], lagoon[1:]):
    layout[min(yf,yt):max(yf,yt)+1, min(xf,xt):max(xf,xt)+1] = 2
layout = np.pad(layout,1)
while np.any(layout != (layout:=evolve(layout))):  pass

np.sum(np.where(layout[1:-1,1:-1]>0,
                np.expand_dims(gaps[0],1)*np.expand_dims(gaps[1],0),
                0)).item()

83605563360288